# DP2 — DiaObject / DiaSource catalog query over a DDF via Butler

**Author:** dagoret  
**Date:** 2026-03-13  

## Purpose

Retrieve **DiaObject**, **DiaSource**, and **ForcedSourceOnDiaObject** catalogs
for a user-selected Deep Drilling Field (DDF) using **Butler Gen3 only**
(no TAP service, not yet available for DP2 at USDF).

## Actual schema (discovered at runtime — ECDFS, v30_0_4)

- `dia_object` returns a DataFrame whose **index** carries the object identifier
  (index name is typically `diaObjectId`).
- The regular columns are: `ra`, `dec`, `nDiaSources`,
  and per-band aggregates `{band}_psfFluxMean/MeanErr/Sigma/Ndata/Min/Max/MaxSlope/ErrMean`
  plus `{band}_scienceFluxMean/MeanErr`.
- There is **no `diaObjectId` regular column** — the helper `to_dataframe()`
  promotes the named index to a regular column before returning.

## Reference notebooks

- `2026-03-07_AccessToDP2.ipynb` — Butler setup, dataset type exploration
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — SkyMap tract lookup

---
## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u

from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

---
## 1. User Configuration

In [ ]:
# =========================================================================
# USER PARAMETERS — edit here
# =========================================================================

SELECTED_DDF = "ECDFS"   # COSMOS | ECDFS | ELAIS-S1 | XMM-LSS | EDFS-a | EDFS-b | EDFS | M49
CONE_RADIUS_DEG = 1.8    # search cone radius around the DDF center (degrees)
MIN_NSOURCES    = 50     # minimum nDiaSources to retain a DiaObject

REPO        = "dp2_prep"
COLLECTION  = "LSSTCam/runs/DRP/v30_0_4/DM-54249"
SKYMAP_NAME = "lsst_cells_v2"

TRACT_SEARCH_RADIUS_DEG = 1.8

DDF_COORDS = {
    "COSMOS"   : (150.119, +2.206),
    "ECDFS"    : ( 53.125, -28.100),
    "ELAIS-S1" : (  9.450, -44.000),
    "XMM-LSS"  : ( 35.708,  -4.750),
    "EDFS-a"   : ( 58.900, -49.315),
    "EDFS-b"   : ( 63.600, -47.600),
    "EDFS"     : ( 61.240, -48.423),
    "M49"      : (187.400,  +8.000),
}

RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]
print(f"DDF         : {SELECTED_DDF}  ({RA_CENTER:.4f}°, {DEC_CENTER:+.4f}°)")
print(f"Cone radius : {CONE_RADIUS_DEG}°")
print(f"Min nDiaSrc : {MIN_NSOURCES}")
print(f"Collection  : {COLLECTION}")

---
## 2. Butler and SkyMap

In [ ]:
butler   = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

---
## 3. Introspect Dataset Types

In [ ]:
SKIP = ('_config', '_log', '_metadata', '_resource_usage', 'Plot', 'Metric', 'metric')
science_types = sorted(
    dt.name for dt in registry.queryDatasetTypes()
    if not any(s in dt.name for s in SKIP)
    and registry.queryDatasets(dt, collections=COLLECTION).any(execute=False, exact=False)
)
print(f"Science dataset types ({len(science_types)}):")
for n in science_types:
    print(f"  {n}")

In [ ]:
for dt_name in ['dia_object', 'dia_source',
                'forced_diff_exp_source_on_dia_object',
                'dia_object_forced_source', 'forcedSourceOnDiaObject']:
    try:
        dt     = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTION).any(
            execute=False, exact=False)
        print(f"{dt_name:50s}  dims={list(dt.dimensions.names)}  in_col={in_col}")
    except Exception as exc:
        print(f"{dt_name:50s}  NOT FOUND ({exc.__class__.__name__})")

In [ ]:
FORCED_ON_DIA_TYPE = None
for candidate in ['forced_diff_exp_source_on_dia_object',
                   'dia_object_forced_source', 'forcedSourceOnDiaObject']:
    try:
        registry.getDatasetType(candidate)
        if registry.queryDatasets(candidate, collections=COLLECTION).any(
                execute=False, exact=False):
            FORCED_ON_DIA_TYPE = candidate
            break
    except Exception:
        pass
print(f"Forced-source type : {FORCED_ON_DIA_TYPE}")

---
## 4. Find Tracts Covering the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec   = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step      = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s  = ra_deg  + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)

tract_ids = find_tracts_for_coord(
    skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG
)
print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

---
## 5. Inspect the Raw Butler Object (schema discovery)

Before loading everything, we inspect **one** reference to understand the exact
schema: index name, index dtype, and column list.

In [ ]:
# -------------------------------------------------------------------------
# Load one reference to discover the real schema
# -------------------------------------------------------------------------
refs_probe = list(registry.queryDatasets(
    "dia_object",
    collections=COLLECTION,
    dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
))
assert refs_probe, "No dia_object reference found for the first tract — check collection."

obj_raw = butler.get(refs_probe[0])

# Convert to pandas without any index manipulation yet
if hasattr(obj_raw, 'to_pandas'):
    df_probe = obj_raw.to_pandas()
elif hasattr(obj_raw, 'to_table'):
    df_probe = obj_raw.to_table().to_pandas()
elif isinstance(obj_raw, Table):
    df_probe = obj_raw.to_pandas()
else:
    df_probe = obj_raw

print("=== Raw DataFrame schema ===")
print(f"index.name  : {df_probe.index.name!r}")
print(f"index.dtype : {df_probe.index.dtype}")
print(f"index[:3]   : {df_probe.index[:3].tolist()}")
print(f"columns     : {list(df_probe.columns)}")
print(f"shape       : {df_probe.shape}")

In [ ]:
# -------------------------------------------------------------------------
# Determine the ID column name from the probe
# -------------------------------------------------------------------------

# Case 1 — index has a name (most common: 'diaObjectId')
if df_probe.index.name is not None:
    OBJ_ID_COL = df_probe.index.name
    ID_IN_INDEX = True
    print(f"Object ID is in the index: '{OBJ_ID_COL}'")

# Case 2 — ID already a regular column
elif any(c in df_probe.columns for c in ['diaObjectId', 'dia_object_id', 'objectId']):
    OBJ_ID_COL = next(c for c in ['diaObjectId', 'dia_object_id', 'objectId']
                      if c in df_probe.columns)
    ID_IN_INDEX = False
    print(f"Object ID is a regular column: '{OBJ_ID_COL}'")

# Case 3 — no ID found, use integer row number
else:
    OBJ_ID_COL  = 'row_id'
    ID_IN_INDEX = False
    print("WARNING: no ID found — will use sequential row numbers as 'row_id'.")

print(f"\nOBJ_ID_COL  = '{OBJ_ID_COL}'")
print(f"ID_IN_INDEX = {ID_IN_INDEX}")

---
## 6. Load DiaObject Catalog

In [ ]:
def to_dataframe(obj):
    """
    Convert a Butler catalog object to a pandas DataFrame.

    If the object ID lives in the index (ID_IN_INDEX=True), the index is
    promoted to a regular column named OBJ_ID_COL before returning.
    This relies on the globals OBJ_ID_COL and ID_IN_INDEX set in Section 5.
    """
    # Step 1 — convert to raw pandas DataFrame
    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, 'to_pandas'):
        df = obj.to_pandas()
    elif hasattr(obj, 'to_table'):
        df = obj.to_table().to_pandas()
    elif isinstance(obj, Table):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Cannot convert {type(obj)} to DataFrame.")

    # Step 2 — promote index → column if needed
    if ID_IN_INDEX and OBJ_ID_COL not in df.columns:
        # Keep the index values as a new column, then replace index with RangeIndex
        df.insert(0, OBJ_ID_COL, df.index)
        df = df.reset_index(drop=True)

    # Case 3 — no ID at all: assign sequential row numbers
    elif OBJ_ID_COL == 'row_id' and OBJ_ID_COL not in df.columns:
        df.insert(0, OBJ_ID_COL, range(len(df)))

    return df

In [ ]:
# -------------------------------------------------------------------------
# Load dia_object for all tracts, concatenate
# -------------------------------------------------------------------------
dia_obj_frames = []

for tract_id in tract_ids:
    dataId = {"skymap": SKYMAP_NAME, "tract": tract_id}
    refs   = list(registry.queryDatasets(
        "dia_object", collections=COLLECTION, dataId=dataId))
    for ref in refs:
        try:
            obj    = butler.get(ref)
            df_tmp = to_dataframe(obj)
            df_tmp['_tract'] = tract_id
            dia_obj_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaObjects")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

if not dia_obj_frames:
    raise RuntimeError("No dia_object data loaded.")

df_dia_obj_all = pd.concat(dia_obj_frames, ignore_index=True)

# Drop cross-tract duplicates on the ID column
n_before = len(df_dia_obj_all)
df_dia_obj_all = df_dia_obj_all.drop_duplicates(subset=OBJ_ID_COL)
print(f"\nTotal DiaObjects: {n_before:,} → {len(df_dia_obj_all):,} after dedup")
print(f"Columns: {list(df_dia_obj_all.columns)[:8]} ...")

In [ ]:
# -------------------------------------------------------------------------
# Detect RA/Dec, nDiaSources, and available bands
# -------------------------------------------------------------------------
RA_COL, DEC_COL = None, None
for ra_c, dec_c in [('ra', 'dec'), ('coord_ra', 'coord_dec'), ('RA', 'DEC')]:
    if ra_c in df_dia_obj_all.columns and dec_c in df_dia_obj_all.columns:
        RA_COL, DEC_COL = ra_c, dec_c
        break
if RA_COL is None:
    raise RuntimeError(f"No RA/Dec found. Columns: {list(df_dia_obj_all.columns)}")

NSRC_COL = next(
    (c for c in ['nDiaSources', 'n_dia_sources', 'ndiasources']
     if c in df_dia_obj_all.columns), None
)

# Bands inferred from *_psfFluxNdata columns
BANDS = sorted({
    c.split('_')[0] for c in df_dia_obj_all.columns
    if c.endswith('_psfFluxNdata') and len(c.split('_')[0]) == 1
})

print(f"ID col    : '{OBJ_ID_COL}'")
print(f"RA col    : '{RA_COL}'")
print(f"Dec col   : '{DEC_COL}'")
print(f"nSrc col  : '{NSRC_COL}'")
print(f"Bands     : {BANDS}")

In [ ]:
# -------------------------------------------------------------------------
# Spatial cone cut
# -------------------------------------------------------------------------
center_sky = SkyCoord(ra=RA_CENTER * u.deg, dec=DEC_CENTER * u.deg)
obj_sky    = SkyCoord(
    ra  = df_dia_obj_all[RA_COL].values  * u.deg,
    dec = df_dia_obj_all[DEC_COL].values * u.deg,
)
sep_deg = center_sky.separation(obj_sky).deg

mask_cone       = sep_deg <= CONE_RADIUS_DEG
df_dia_obj_cone = df_dia_obj_all[mask_cone].copy()
df_dia_obj_cone['_sep_deg'] = sep_deg[mask_cone]

print(f"DiaObjects in cone ({CONE_RADIUS_DEG}°): "
      f"{len(df_dia_obj_cone):,} / {len(df_dia_obj_all):,}")

In [ ]:
# -------------------------------------------------------------------------
# Filter: nDiaSources >= MIN_NSOURCES
# -------------------------------------------------------------------------
if NSRC_COL:
    df_dia_obj_rich = (
        df_dia_obj_cone[df_dia_obj_cone[NSRC_COL] >= MIN_NSOURCES]
        .sort_values(NSRC_COL, ascending=False)
        .reset_index(drop=True)
    )
else:
    print("WARNING: nDiaSources not found — keeping all cone objects.")
    df_dia_obj_rich = df_dia_obj_cone.copy()

print(f"DiaObjects with {NSRC_COL} >= {MIN_NSOURCES}: {len(df_dia_obj_rich):,}")
if len(df_dia_obj_rich) > 0 and NSRC_COL:
    print(f"  max    : {df_dia_obj_rich[NSRC_COL].max()}")
    print(f"  median : {df_dia_obj_rich[NSRC_COL].median():.1f}")

In [ ]:
# Preview
peek_cols = [OBJ_ID_COL, RA_COL, DEC_COL, NSRC_COL, '_tract', '_sep_deg']
peek_cols = [c for c in peek_cols if c in df_dia_obj_rich.columns]
display(df_dia_obj_rich[peek_cols].head(20))

---
## 7. Diagnostic Plots

In [ ]:
BAND_COLORS = {'u': 'purple', 'g': 'green', 'r': 'red',
               'i': 'darkorange', 'z': 'sienna', 'y': 'black'}

# =========================================================================
# Figure 1 — nDiaSources distribution
# =========================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
if NSRC_COL:
    ax.hist(df_dia_obj_cone[NSRC_COL].dropna(), bins=100, log=True,
            color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(MIN_NSOURCES, color='red', linestyle='--', linewidth=1.5,
               label=f'cut ≥ {MIN_NSOURCES}')
ax.set_xlabel('nDiaSources', fontsize=11)
ax.set_ylabel('N DiaObjects', fontsize=11)
ax.set_title(f'{SELECTED_DDF} — All in cone (N={len(df_dia_obj_cone):,})',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
if NSRC_COL and len(df_dia_obj_rich) > 0:
    ax.hist(df_dia_obj_rich[NSRC_COL].dropna(), bins=50,
            color='darkorange', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{SELECTED_DDF} — Filtered (N={len(df_dia_obj_rich):,})',
                 fontsize=11, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No objects pass the cut',
            transform=ax.transAxes, ha='center', fontsize=13)
ax.set_xlabel('nDiaSources', fontsize=11)
ax.set_ylabel('N DiaObjects', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'DiaObj_nSources_hist_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =========================================================================
# Figure 2 — Sky distribution
# =========================================================================
fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor('#f5f5f5')
ax.invert_xaxis()

ax.scatter(df_dia_obj_cone[RA_COL], df_dia_obj_cone[DEC_COL],
           s=1, color='lightgrey', alpha=0.4, zorder=1,
           label=f'All in cone ({len(df_dia_obj_cone):,})')

if NSRC_COL and len(df_dia_obj_rich) > 0:
    sc = ax.scatter(
        df_dia_obj_rich[RA_COL], df_dia_obj_rich[DEC_COL],
        c=df_dia_obj_rich[NSRC_COL],
        cmap='plasma', s=14, alpha=0.9, zorder=3,
        norm=mcolors.LogNorm(
            vmin=df_dia_obj_rich[NSRC_COL].min(),
            vmax=df_dia_obj_rich[NSRC_COL].max()),
        label=f'{NSRC_COL} ≥ {MIN_NSOURCES} ({len(df_dia_obj_rich):,})'
    )
    plt.colorbar(sc, ax=ax, pad=0.02).set_label('nDiaSources (log)', fontsize=10)

ax.plot(RA_CENTER, DEC_CENTER, marker='*', markersize=18, color='gold',
        markeredgecolor='black', markeredgewidth=1.0,
        zorder=10, linestyle='none', label=f'{SELECTED_DDF} center')

cos_dec = np.cos(np.deg2rad(DEC_CENTER))
theta   = np.linspace(0, 2 * np.pi, 361)
ax.plot(RA_CENTER + CONE_RADIUS_DEG / cos_dec * np.cos(theta),
        DEC_CENTER + CONE_RADIUS_DEG * np.sin(theta),
        color='red', linewidth=1.2, linestyle='--', zorder=4,
        label=f'Cone {CONE_RADIUS_DEG}°')

ax.set_xlabel('RA (deg)', fontsize=12)
ax.set_ylabel('Dec (deg)', fontsize=12)
ax.set_title(f'DP2 DiaObjects — {SELECTED_DDF}  (tracts {tract_ids})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f'DiaObj_skymap_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Per-Band Summary Statistics

Fluxes are pre-aggregated in `dia_object` — no DiaSource load needed for this.

In [ ]:
# =========================================================================
# Figure 3 — Mean PSF flux vs sigma per band (top-N objects)
# =========================================================================
TOP_N  = min(500, len(df_dia_obj_rich))
df_top = df_dia_obj_rich.head(TOP_N)

ncols = 3
nrows = int(np.ceil(len(BANDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten()

for idx, band in enumerate(BANDS):
    ax        = axes[idx]
    mean_col  = f'{band}_psfFluxMean'
    sigma_col = f'{band}_psfFluxSigma'
    ndata_col = f'{band}_psfFluxNdata'

    if mean_col not in df_top.columns:
        ax.set_visible(False)
        continue

    valid = df_top[mean_col].notna() & (df_top[ndata_col] > 0)
    x = df_top.loc[valid, mean_col]
    y = df_top.loc[valid, sigma_col] if sigma_col in df_top.columns else np.zeros(valid.sum())
    c = df_top.loc[valid, ndata_col] if ndata_col in df_top.columns else 'grey'

    sc = ax.scatter(x, y, c=c, cmap='viridis', s=8, alpha=0.7)
    plt.colorbar(sc, ax=ax, label='nData (epochs)')
    ax.set_xlabel(f'{band} psfFluxMean (nJy)', fontsize=9)
    ax.set_ylabel(f'{band} psfFluxSigma (nJy)', fontsize=9)
    ax.set_title(f'Band {band}  (N={valid.sum():,})',
                 fontsize=10, fontweight='bold',
                 color=BAND_COLORS.get(band, 'black'))
    ax.grid(True, alpha=0.3)

for idx in range(len(BANDS), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    f'{SELECTED_DDF} — PSF flux mean vs sigma per band\n'
    f'(top {TOP_N} objects by nDiaSources)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'DiaObj_flux_mean_sigma_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =========================================================================
# Figure 4 — Number of epochs per band distribution
# =========================================================================
fig, ax = plt.subplots(figsize=(10, 5))
for band in BANDS:
    ndata_col = f'{band}_psfFluxNdata'
    if ndata_col in df_dia_obj_rich.columns:
        vals = df_dia_obj_rich[ndata_col].dropna()
        ax.hist(vals, bins=50, alpha=0.55,
                color=BAND_COLORS.get(band, 'grey'),
                label=f'{band} (med={vals.median():.0f})',
                histtype='stepfilled')
ax.set_xlabel('psfFluxNdata (epochs per band)', fontsize=11)
ax.set_ylabel('N DiaObjects', fontsize=11)
ax.set_title(
    f'{SELECTED_DDF} — Epochs per band  ({NSRC_COL} ≥ {MIN_NSOURCES}, N={len(df_dia_obj_rich):,})',
    fontsize=11, fontweight='bold'
)
ax.legend(title='Band', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'DiaObj_ndata_per_band_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Load DiaSources (individual detections)

In [ ]:
# Set of object IDs to keep (used for filtering DiaSources and forced photometry)
rich_ids = set(df_dia_obj_rich[OBJ_ID_COL].values)
print(f"Filtering on {len(rich_ids):,} object IDs (column '{OBJ_ID_COL}')")

In [ ]:
# -------------------------------------------------------------------------
# Load dia_source per tract
# -------------------------------------------------------------------------
dia_src_frames = []

for tract_id in tract_ids:
    dataId = {"skymap": SKYMAP_NAME, "tract": tract_id}
    refs   = list(registry.queryDatasets(
        "dia_source", collections=COLLECTION, dataId=dataId))
    for ref in refs:
        try:
            obj    = butler.get(ref)
            df_tmp = to_dataframe(obj)
            # Find the join column in this table
            join_col = next(
                (c for c in [OBJ_ID_COL, 'diaObjectId', 'dia_object_id']
                 if c in df_tmp.columns),
                None
            )
            if join_col:
                df_tmp = df_tmp[df_tmp[join_col].isin(rich_ids)]
            else:
                print(f"  WARNING: no join column found in dia_source "
                      f"(tract {tract_id}). Columns: {list(df_tmp.columns[:10])}")
            dia_src_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaSources  "
                  f"cols[:8]: {list(df_tmp.columns[:8])}")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

if dia_src_frames:
    df_dia_src_rich = pd.concat(dia_src_frames, ignore_index=True)
    src_id_col = next(
        (c for c in ['diaSourceId', 'dia_source_id'] if c in df_dia_src_rich.columns),
        None
    )
    if src_id_col:
        df_dia_src_rich = df_dia_src_rich.drop_duplicates(subset=src_id_col)
    print(f"\nTotal DiaSources (filtered): {len(df_dia_src_rich):,}")
    print(f"All columns: {list(df_dia_src_rich.columns)}")
else:
    print("No DiaSources loaded.")
    df_dia_src_rich = pd.DataFrame()

In [ ]:
if len(df_dia_src_rich) > 0:
    MJD_COL  = next((c for c in ['midPointMjdTai', 'midpointMjdTai', 'mjd']
                      if c in df_dia_src_rich.columns), None)
    BAND_COL = next((c for c in ['band', 'filterName', 'filter']
                      if c in df_dia_src_rich.columns), None)
    FLUX_COL = next((c for c in ['psfFlux', 'psf_flux']
                      if c in df_dia_src_rich.columns), None)
    FERR_COL = next((c for c in ['psfFluxErr', 'psf_flux_err']
                      if c in df_dia_src_rich.columns), None)
    print(f"MJD : {MJD_COL}  |  Band : {BAND_COL}  |  Flux : {FLUX_COL} ± {FERR_COL}")
else:
    MJD_COL = BAND_COL = FLUX_COL = FERR_COL = None

In [ ]:
# =========================================================================
# Figure 5 — Flux vs MJD scatter for all DiaSources
# =========================================================================
if MJD_COL and FLUX_COL and BAND_COL and len(df_dia_src_rich) > 0:
    fig, ax = plt.subplots(figsize=(13, 5))
    for band, grp in df_dia_src_rich.groupby(BAND_COL):
        ax.scatter(grp[MJD_COL], grp[FLUX_COL], s=1, alpha=0.2,
                   color=BAND_COLORS.get(str(band), 'grey'), label=str(band))
    ax.set_xlabel('MJD (TAI)', fontsize=12)
    ax.set_ylabel('PSF flux (nJy)', fontsize=12)
    ax.set_title(
        f'All DiaSources — {SELECTED_DDF}  '
        f'({NSRC_COL} ≥ {MIN_NSOURCES},  N_src={len(df_dia_src_rich):,})',
        fontsize=12, fontweight='bold')
    ax.legend(title='Band', fontsize=9, markerscale=6)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'DiaSrc_scatter_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("DiaSources not available or column names not resolved.")

---
## 10. Load Forced Photometry

In [ ]:
df_forced_rich = pd.DataFrame()

if FORCED_ON_DIA_TYPE is None:
    print("No forced-source dataset type found — skipping.")
else:
    forced_frames = []
    for tract_id in tract_ids:
        dataId = {"skymap": SKYMAP_NAME, "tract": tract_id}
        refs   = list(registry.queryDatasets(
            FORCED_ON_DIA_TYPE, collections=COLLECTION, dataId=dataId))
        for ref in refs:
            try:
                obj    = butler.get(ref)
                df_tmp = to_dataframe(obj)
                join_col = next(
                    (c for c in [OBJ_ID_COL, 'diaObjectId', 'dia_object_id']
                     if c in df_tmp.columns), None)
                if join_col:
                    df_tmp = df_tmp[df_tmp[join_col].isin(rich_ids)]
                forced_frames.append(df_tmp)
                print(f"  Tract {tract_id:6d} — {len(df_tmp):,} forced rows  "
                      f"cols[:8]: {list(df_tmp.columns[:8])}")
            except Exception as exc:
                print(f"  Tract {tract_id:6d} — ERROR: {exc}")

    if forced_frames:
        df_forced_rich = pd.concat(forced_frames, ignore_index=True)
        print(f"\nTotal forced rows: {len(df_forced_rich):,}")
        print(f"All columns: {list(df_forced_rich.columns)}")
    else:
        print("No forced photometry loaded.")

In [ ]:
if len(df_forced_rich) > 0:
    FMJD_COL  = next((c for c in ['midpointMjdTai', 'midPointMjdTai', 'mjd']
                       if c in df_forced_rich.columns), None)
    FBAND_COL = next((c for c in ['band', 'filterName', 'filter']
                       if c in df_forced_rich.columns), None)
    FFLUX_COL = next((c for c in ['psfFlux', 'psf_flux']
                       if c in df_forced_rich.columns), None)
    FFERR_COL = next((c for c in ['psfFluxErr', 'psf_flux_err']
                       if c in df_forced_rich.columns), None)
    FDIFF_COL = next((c for c in ['psfDiffFlux', 'psf_diff_flux']
                       if c in df_forced_rich.columns), None)
    FDERR_COL = next((c for c in ['psfDiffFluxErr', 'psf_diff_flux_err']
                       if c in df_forced_rich.columns), None)
    FOBJ_COL  = next(
        (c for c in [OBJ_ID_COL, 'diaObjectId', 'dia_object_id']
         if c in df_forced_rich.columns), None)
    print(f"MJD   : {FMJD_COL}  |  band : {FBAND_COL}")
    print(f"Flux  : {FFLUX_COL} ± {FFERR_COL}")
    print(f"Diff  : {FDIFF_COL} ± {FDERR_COL}")
    print(f"ObjID : {FOBJ_COL}")
else:
    FMJD_COL = FBAND_COL = FFLUX_COL = FFERR_COL = None
    FDIFF_COL = FDERR_COL = FOBJ_COL = None

In [ ]:
# =========================================================================
# Figure 6 — Forced LC of the most-detected DiaObject
# =========================================================================
if len(df_forced_rich) > 0 and FMJD_COL and FOBJ_COL:
    top_id = df_dia_obj_rich.iloc[0][OBJ_ID_COL]
    top_n  = df_dia_obj_rich.iloc[0][NSRC_COL] if NSRC_COL else 'N/A'
    df_lc  = df_forced_rich[df_forced_rich[FOBJ_COL] == top_id].sort_values(FMJD_COL)

    nrows = 2 if (FDIFF_COL and FDIFF_COL in df_lc.columns) else 1
    fig, axes = plt.subplots(nrows, 1, figsize=(13, 5 * nrows), sharex=True)
    if nrows == 1:
        axes = [axes]

    ax = axes[0]
    for band, grp in df_lc.groupby(FBAND_COL):
        yerr = grp[FFERR_COL] if FFERR_COL else None
        ax.errorbar(grp[FMJD_COL], grp[FFLUX_COL], yerr=yerr,
                    fmt='o', ms=4, alpha=0.8, capsize=2,
                    color=BAND_COLORS.get(str(band), 'grey'), label=str(band))
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Science PSF flux (nJy)', fontsize=11)
    ax.set_title(
        f'Forced LC — {OBJ_ID_COL}={top_id}  |  nDiaSrc={top_n}\n{SELECTED_DDF}',
        fontsize=12, fontweight='bold')
    ax.legend(title='Band', fontsize=9)
    ax.grid(True, alpha=0.3)

    if nrows == 2:
        ax = axes[1]
        for band, grp in df_lc.groupby(FBAND_COL):
            yerr = grp[FDERR_COL] if FDERR_COL else None
            ax.errorbar(grp[FMJD_COL], grp[FDIFF_COL], yerr=yerr,
                        fmt='o', ms=4, alpha=0.8, capsize=2,
                        color=BAND_COLORS.get(str(band), 'grey'), label=str(band))
        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_ylabel('Diff image PSF flux (nJy)', fontsize=11)
        ax.legend(title='Band', fontsize=9)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('MJD (TAI)', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'DiaObj_forcedLC_{top_id}_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Forced LC plot skipped (no forced photometry or column names not resolved).")

---
## 11. Coordinate Cross-Match with a Fink Alert

DRP and Alert Production assign independent IDs — cross-match by (RA, Dec) only.

In [ ]:
# =========================================================================
# User-supplied Fink alert position — edit here
# =========================================================================
FINK_RA              = 53.125   # degrees ICRS
FINK_DEC             = -28.100  # degrees ICRS
XMATCH_RADIUS_ARCSEC = 1.0

print(f"Fink : RA={FINK_RA:.6f}°  Dec={FINK_DEC:+.6f}°  r={XMATCH_RADIUS_ARCSEC}\"")

In [ ]:
alert_sky  = SkyCoord(ra=FINK_RA * u.deg, dec=FINK_DEC * u.deg)
all_sky    = SkyCoord(
    ra  = df_dia_obj_cone[RA_COL].values  * u.deg,
    dec = df_dia_obj_cone[DEC_COL].values * u.deg,
)
sep_arcsec = alert_sky.separation(all_sky).arcsec

df_xmatch = df_dia_obj_cone.copy()
df_xmatch['_sep_arcsec'] = sep_arcsec
df_xmatch_cut = (
    df_xmatch[df_xmatch['_sep_arcsec'] <= XMATCH_RADIUS_ARCSEC]
    .sort_values('_sep_arcsec')
    .reset_index(drop=True)
)

print(f"Candidates within {XMATCH_RADIUS_ARCSEC}\": {len(df_xmatch_cut)}")
if len(df_xmatch_cut) == 0:
    print("No match — try increasing XMATCH_RADIUS_ARCSEC.")
else:
    show_cols = [OBJ_ID_COL, RA_COL, DEC_COL, '_sep_arcsec']
    if NSRC_COL:
        show_cols.append(NSRC_COL)
    display(df_xmatch_cut[[c for c in show_cols if c in df_xmatch_cut.columns]].head(10))
    best_id  = df_xmatch_cut.iloc[0][OBJ_ID_COL]
    best_sep = df_xmatch_cut.iloc[0]['_sep_arcsec']
    print(f"Best match: {OBJ_ID_COL}={best_id}  sep={best_sep:.4f}\"")

In [ ]:
# Forced LC of the cross-matched object
if len(df_xmatch_cut) > 0 and len(df_forced_rich) > 0 and FOBJ_COL and FMJD_COL:
    best_id  = df_xmatch_cut.iloc[0][OBJ_ID_COL]
    best_sep = df_xmatch_cut.iloc[0]['_sep_arcsec']
    df_lc = df_forced_rich[df_forced_rich[FOBJ_COL] == best_id].sort_values(FMJD_COL)

    nrows = 2 if (FDIFF_COL and FDIFF_COL in df_lc.columns) else 1
    fig, axes = plt.subplots(nrows, 1, figsize=(13, 5 * nrows), sharex=True)
    if nrows == 1:
        axes = [axes]

    ax = axes[0]
    for band, grp in df_lc.groupby(FBAND_COL):
        yerr = grp[FFERR_COL] if FFERR_COL else None
        ax.errorbar(grp[FMJD_COL], grp[FFLUX_COL], yerr=yerr,
                    fmt='o', ms=4, alpha=0.85, capsize=2,
                    color=BAND_COLORS.get(str(band), 'grey'), label=str(band))
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_ylabel('PSF flux (nJy)', fontsize=11)
    ax.set_title(
        f'Cross-matched DRP LC — {OBJ_ID_COL}={best_id}  sep={best_sep:.3f}"\n'
        f'Fink: RA={FINK_RA:.5f}°  Dec={FINK_DEC:+.5f}°',
        fontsize=11, fontweight='bold')
    ax.legend(title='Band', fontsize=9)
    ax.grid(True, alpha=0.3)

    if nrows == 2:
        ax = axes[1]
        for band, grp in df_lc.groupby(FBAND_COL):
            yerr = grp[FDERR_COL] if FDERR_COL else None
            ax.errorbar(grp[FMJD_COL], grp[FDIFF_COL], yerr=yerr,
                        fmt='o', ms=4, alpha=0.85, capsize=2,
                        color=BAND_COLORS.get(str(band), 'grey'), label=str(band))
        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_ylabel('Diff image PSF flux (nJy)', fontsize=11)
        ax.legend(title='Band', fontsize=9)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('MJD (TAI)', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'DiaObj_xmatch_LC_{best_id}_{SELECTED_DDF}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Cross-match LC skipped (no match or no forced photometry).")

---
## 12. Save Results

In [ ]:
out = f"DiaObjects_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
df_dia_obj_rich.to_csv(out, index=False)
print(f"Saved: {out}  ({len(df_dia_obj_rich):,} rows)")

if len(df_dia_src_rich) > 0:
    out = f"DiaSources_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
    df_dia_src_rich.to_csv(out, index=False)
    print(f"Saved: {out}  ({len(df_dia_src_rich):,} rows)")

if len(df_forced_rich) > 0:
    out = f"ForcedSrc_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
    df_forced_rich.to_csv(out, index=False)
    print(f"Saved: {out}  ({len(df_forced_rich):,} rows)")